# Week 4: FinBERT Fusion Model Training

**Goal:** Fine-tune a `FinBertFusionClassifier` (frozen FinBERT backbone + price feature fusion) on the
financial corpus and export the checkpoint for integration into `src/forecast_model.py`.

**Environment:** Google Colab T4 GPU

**Steps:**
1. Mount Google Drive and install dependencies
2. Load `data/financial_corpus.csv` (or `data/sample_dataset.json`)
3. Tokenise headlines with `ProsusAI/finbert`
4. Train `FinBertFusionClassifier` for 5 epochs
5. Evaluate and compare vs rule-based baseline
6. Export `models/finbert_fusion.pt` + `models/label_encoder.pkl`

## 0. Mount Drive & Install Dependencies

In [ ]:
# ── Run only on Colab ──────────────────────────────────────────────────────
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust this path to where you cloned / uploaded the project
    PROJECT_ROOT = '/content/drive/MyDrive/Agentic-AI-in-SDLC'
else:
    import pathlib
    PROJECT_ROOT = str(pathlib.Path('..').resolve())

print('Project root:', PROJECT_ROOT)

In [ ]:
# Install GPU dependencies (idempotent)
!pip install -q torch transformers scikit-learn accelerate pandas numpy

## 1. Imports & Configuration

In [ ]:
import json
import os
import pathlib
import pickle
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, random_split
from transformers import AutoModel, AutoTokenizer

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR   = pathlib.Path(PROJECT_ROOT) / 'data'
MODELS_DIR = pathlib.Path(PROJECT_ROOT) / 'models'
SRC_DIR    = pathlib.Path(PROJECT_ROOT) / 'src'
MODELS_DIR.mkdir(exist_ok=True)

CHECKPOINT_PATH     = MODELS_DIR / 'finbert_fusion.pt'
LABEL_ENCODER_PATH  = MODELS_DIR / 'label_encoder.pkl'

# ── Hyper-parameters ───────────────────────────────────────────────────────
FINBERT_MODEL = 'ProsusAI/finbert'
MAX_LEN       = 128
BATCH_SIZE    = 16
EPOCHS        = 5
LR            = 2e-4
VAL_SPLIT     = 0.20
LABEL_NAMES   = ['DOWN', 'HOLD', 'UP']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 2. Load & Pre-process Data

In [ ]:
def load_corpus(data_dir: pathlib.Path) -> pd.DataFrame:
    """Load dataset from CSV or JSON fallback, return a flat DataFrame."""
    csv_path = data_dir / 'financial_corpus.csv'
    json_path = data_dir / 'sample_dataset.json'

    if csv_path.exists():
        df = pd.read_csv(csv_path)
        # Expected columns: ticker, headline / title / cleaned_text,
        #                   price_5d_return, volume_change_pct, label
        print(f'Loaded CSV: {len(df)} rows')
    elif json_path.exists():
        with json_path.open(encoding='utf-8') as f:
            records = json.load(f)
        def _safe_float(val, default=0.0):
            """Convert val to float; return default for None / non-numeric strings."""
            result = pd.to_numeric(val, errors='coerce')
            return float(result) if result == result else default  # NaN != NaN

        rows = []
        for rec in records:
            label = rec.get('label') or rec.get('ground_truth', {}).get('label', 'HOLD')
            pf = rec.get('price_features', {})
            for news in rec.get('news_data', rec.get('news', [])):
                headline = (
                    news.get('cleaned_text') or
                    news.get('title') or
                    news.get('raw_title', '')
                )
                rows.append({
                    'headline': (str(headline).strip() or '[no headline]').lower(),
                    'price_5d_return': _safe_float(pf.get('price_5d_return')),
                    'volume_change_pct': _safe_float(pf.get('volume_change_pct')),
                    'label': str(label).upper(),
                })
        df = pd.DataFrame(rows)
        print(f'Loaded JSON fallback: {len(df)} rows')
    else:
        raise FileNotFoundError(
            f'No data found in {data_dir}. '
            'Expected financial_corpus.csv or sample_dataset.json.'
        )

    # Normalise column names
    for col in ('headline', 'title', 'raw_title', 'cleaned_text'):
        if col in df.columns and 'headline' not in df.columns:
            df = df.rename(columns={col: 'headline'})
            break
    if 'headline' not in df.columns:
        raise ValueError('Cannot find a headline / text column in dataset.')

    df['headline'] = df['headline'].fillna('').astype(str).str.lower()
    df['price_5d_return'] = pd.to_numeric(df.get('price_5d_return', 0), errors='coerce').fillna(0.0)
    df['volume_change_pct'] = pd.to_numeric(df.get('volume_change_pct', 0), errors='coerce').fillna(0.0)
    df['label'] = df['label'].str.upper().str.strip()
    df = df[df['label'].isin(LABEL_NAMES)].reset_index(drop=True)
    return df


df = load_corpus(DATA_DIR)
print('Label distribution:')
print(df['label'].value_counts())
df.head()

## 3. Label Encoding

In [ ]:
le = LabelEncoder()
le.fit(LABEL_NAMES)  # fixed order: DOWN=0, HOLD=1, UP=2
df['label_id'] = le.transform(df['label'])

# Save for inference-time decoding
with open(LABEL_ENCODER_PATH, 'wb') as f:
    pickle.dump(le, f)
print('Label encoder saved to', LABEL_ENCODER_PATH)
print('Classes:', list(le.classes_))

## 4. PyTorch Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)


class FinBERTDataset(Dataset):
    """Tokenises financial headlines and packages them with price features."""

    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_len: int = MAX_LEN):
        self.headlines = dataframe['headline'].tolist()
        self.price_5d  = dataframe['price_5d_return'].tolist()
        self.vol_chg   = dataframe['volume_change_pct'].tolist()
        self.labels    = dataframe['label_id'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.headlines)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.headlines[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        price_feats = torch.tensor(
            [self.price_5d[idx], self.vol_chg[idx]], dtype=torch.float32
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'price_features': price_feats,
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
        }


full_dataset = FinBERTDataset(df, tokenizer)
val_size   = max(1, int(len(full_dataset) * VAL_SPLIT))
train_size = len(full_dataset) - val_size
train_ds, val_ds = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
print(f'Train: {train_size} | Val: {val_size}')

## 5. Model Definition

In [ ]:
class FinBertFusionClassifier(nn.Module):
    """
    Frozen FinBERT backbone + price feature fusion head.

    Architecture:
        CLS (768) ++ price_feats (2)  →  Linear(770, 128)  →  ReLU  →  Linear(128, 3)
    """

    def __init__(self, freeze_bert: bool = True, num_classes: int = 3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(FINBERT_MODEL, use_safetensors=True)
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        bert_hidden = self.bert.config.hidden_size  # 768 for FinBERT
        self.fusion  = nn.Linear(bert_hidden + 2, 128)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.output  = nn.Linear(128, num_classes)

    def forward(self, input_ids, attention_mask, price_features):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_vec  = bert_out.last_hidden_state[:, 0, :]          # (B, 768)
        fused    = torch.cat([cls_vec, price_features], dim=1)  # (B, 770)
        x = self.dropout(self.relu(self.fusion(fused)))          # (B, 128)
        logits = self.output(x)                                  # (B, 3)
        return logits


model = FinBertFusionClassifier(freeze_bert=True).to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

## 6. Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)


def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            price_features = batch['price_features'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)

            logits = model(input_ids, attention_mask, price_features)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
            preds = logits.argmax(dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().tolist())
    avg_loss = total_loss / max(len(loader), 1)
    acc      = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_labels, all_preds


best_val_acc = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        price_features = batch['price_features'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask, price_features)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= max(len(train_loader), 1)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)

    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'val_acc': val_acc})
    print(f'Epoch {epoch}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')

    # Save best checkpoint
    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f'  → checkpoint saved (val_acc={val_acc:.4f})')

print(f'\nBest val accuracy: {best_val_acc:.4f}')
print(f'Checkpoint: {CHECKPOINT_PATH}')

## 7. Final Evaluation

In [ ]:
# Load best checkpoint for final evaluation
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True))
_, val_acc, y_true, y_pred = evaluate(model, val_loader)

print('=== FinBERT Fusion — Validation Results ===')
print(f'Accuracy: {val_acc:.4f}')
print()
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=le.classes_))
print()
print('Confusion Matrix (rows=true, cols=predicted):')
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
print(cm_df)

## 8. Rule-Based vs FinBERT Comparison

In [ ]:
import sys
sys.path.insert(0, str(SRC_DIR))

from forecast_model import forecast_from_news  # rule-based

val_indices = val_ds.indices
val_df = df.iloc[val_indices].reset_index(drop=True)

rule_correct = 0
finbert_correct = 0
n = len(val_df)

for _, row in val_df.iterrows():
    true_label = row['label']
    price_features = {
        'price_5d_return': row['price_5d_return'],
        'volume_change_pct': row['volume_change_pct'],
    }
    # Rule-based
    fake_news_item = [{'title': row['headline'], 'cleaned_text': row['headline']}]
    rb_result = forecast_from_news(fake_news_item, price_features)
    if rb_result['prediction'] == true_label:
        rule_correct += 1

rule_acc = rule_correct / max(n, 1)

comparison = pd.DataFrame({
    'Model': ['Rule-Based', 'FinBERT Fusion'],
    'Accuracy': [f'{rule_acc:.2%}', f'{val_acc:.2%}'],
    'Correct': [rule_correct, int(val_acc * n)],
    'Total': [n, n],
})
print('=== Model Comparison (Validation Set) ===')
print(comparison.to_string(index=False))

## 9. Export Artifacts

After running this cell, download `finbert_fusion.pt` and `label_encoder.pkl` from your Google Drive
and place them in `<project_root>/models/` for local inference.

In [ ]:
print('Artifacts ready for download:')
print(f'  Model checkpoint : {CHECKPOINT_PATH}')
print(f'  Label encoder    : {LABEL_ENCODER_PATH}')

if IN_COLAB:
    from google.colab import files
    print('\nDownloading checkpoint...')
    files.download(str(CHECKPOINT_PATH))
    files.download(str(LABEL_ENCODER_PATH))